In [1]:
import pandas as pd

df = pd.read_csv("../data/processed/logon_cleaned.csv")

df["date"] = pd.to_datetime(df["date"])

print(df.shape)
print(df.head())

(854859, 8)
                         id                date     user       pc activity  \
0  {E0Y1-L6UD60XB-3837XFWY} 2010-01-04 08:09:00  AAE0190  PC-8915    Logon   
1  {B2I9-T5CP00UJ-3212DWJV} 2010-01-04 18:20:00  AAE0190  PC-8915   Logoff   
2  {T0M1-P7SR02LX-8081LEPV} 2010-01-05 08:19:00  AAE0190  PC-8915    Logon   
3  {I0G9-Y8RI12KP-0169VVFX} 2010-01-05 18:11:00  AAE0190  PC-8915   Logoff   
4  {G8L6-G6XS46RU-2239INEZ} 2010-01-06 08:09:00  AAE0190  PC-8915    Logon   

   hour         day  weekday  
0     8  2010-01-04        0  
1    18  2010-01-04        0  
2     8  2010-01-05        1  
3    18  2010-01-05        1  
4     8  2010-01-06        2  


In [2]:
df["hour"] = df["date"].dt.hour
df["weekday"] = df["date"].dt.dayofweek
df["day"] = df["date"].dt.date

In [3]:
df["after_hours"] = (
    (df["hour"] < 8) | (df["hour"] >= 18)
)

df["weekend"] = df["weekday"] >= 5

In [4]:
features = df.groupby(["user", "day"]).agg(
    login_count=("activity", lambda x: (x == "Logon").sum()),
    logout_count=("activity", lambda x: (x == "Logoff").sum()),
    after_hours_count=("after_hours", "sum"),
    weekend_count=("weekend", "sum"),
    unique_pcs=("pc", "nunique"),
    total_events=("activity", "count")
).reset_index()

In [5]:
print(features.shape)
features.head(10)

(330452, 8)


,user,day,login_count,logout_count,after_hours_count,weekend_count,unique_pcs,total_events
0,AAE0190,2010-01-04,1,1,1,0,1,2
1,AAE0190,2010-01-05,1,1,1,0,1,2
2,AAE0190,2010-01-06,1,1,1,0,1,2
3,AAE0190,2010-01-07,1,1,1,0,1,2
4,AAE0190,2010-01-08,1,1,1,0,1,2
5,AAE0190,2010-01-11,1,1,1,0,1,2
6,AAE0190,2010-01-12,1,1,1,0,1,2
7,AAE0190,2010-01-13,1,1,1,0,1,2
8,AAE0190,2010-01-14,1,1,1,0,1,2
9,AAE0190,2010-01-15,1,1,1,0,1,2


In [6]:
features.to_csv(
    "../data/processed/user_daily_features.csv",
    index=False
)

print("Saved user_daily_features.csv")

Saved user_daily_features.csv
